# pose_in_link 的 Plotly 3D 可视化

本 Notebook 将给定的 7 维位姿向量拆分为平移与四元数（`[tx, ty, tz, qx, qy, qz, qw]`），构建旋转矩阵与 4x4 齐次变换矩阵，并使用 `plotly.graph_objects` 进行交互式 3D 可视化。

## 1) 参数定义

In [ ]:
import numpy as np

pose_in_link = np.array([
    0.07783932332093665,
    0.2078814260418823,
    0.34723683952957585,
    0.2273133855008057,
    -0.6785647482083789,
    0.6637673415778982,
    -0.21746591345696367,
], dtype=float)

# 约定：pose_in_link = [tx, ty, tz, qx, qy, qz, qw]（四元数采用 xyzw 顺序）
t = pose_in_link[:3]
q_xyzw = pose_in_link[3:]

print('translation t:', t)
print('quaternion [qx, qy, qz, qw]:', q_xyzw)

## 2) 矩阵构建（手写四元数 -> 旋转矩阵，不依赖 scipy）

In [ ]:
def quat_xyzw_to_rotmat(q_xyzw: np.ndarray) -> np.ndarray:
    """将四元数 [qx, qy, qz, qw] 转为旋转矩阵。"""
    q = np.asarray(q_xyzw, dtype=float).reshape(4,)
    norm = np.linalg.norm(q)
    if norm == 0:
        raise ValueError('Quaternion norm is zero.')

    qx, qy, qz, qw = q / norm

    R = np.array([
        [1 - 2 * (qy * qy + qz * qz),     2 * (qx * qy - qz * qw),     2 * (qx * qz + qy * qw)],
        [    2 * (qx * qy + qz * qw), 1 - 2 * (qx * qx + qz * qz),     2 * (qy * qz - qx * qw)],
        [    2 * (qx * qz - qy * qw),     2 * (qy * qz + qx * qw), 1 - 2 * (qx * qx + qy * qy)],
    ], dtype=float)
    return R

R_target_in_link = quat_xyzw_to_rotmat(q_xyzw)

T_target_in_link = np.eye(4, dtype=float)
T_target_in_link[:3, :3] = R_target_in_link
T_target_in_link[:3, 3] = t

np.set_printoptions(precision=6, suppress=True)
print('R_target_in_link =\n', R_target_in_link)
print('T_target_in_link =\n', T_target_in_link)

## 3) 可视化（Plotly 3D）

In [ ]:
import plotly.graph_objects as go

def add_frame(fig, R, origin, name, axis_len=0.10):
    origin = np.asarray(origin, dtype=float).reshape(3,)
    axes = {
        'x': ('red', R[:, 0]),
        'y': ('green', R[:, 1]),
        'z': ('blue', R[:, 2]),
    }

    for axis_name, (color, direction) in axes.items():
        end = origin + axis_len * direction
        fig.add_trace(go.Scatter3d(
            x=[origin[0], end[0]],
            y=[origin[1], end[1]],
            z=[origin[2], end[2]],
            mode='lines',
            line=dict(color=color, width=7),
            name=f'{name}-{axis_name}',
            legendgroup=name,
            showlegend=True,
        ))

    fig.add_trace(go.Scatter3d(
        x=[origin[0]], y=[origin[1]], z=[origin[2]],
        mode='markers+text',
        marker=dict(size=5, color='black'),
        text=[name],
        textposition='top center',
        name=f'{name}-origin',
        legendgroup=name,
        showlegend=True,
    ))

fig = go.Figure()

# link 坐标系：世界原点处，方向与世界系一致
R_link = np.eye(3)
o_link = np.zeros(3)

# target 坐标系：由 pose_in_link 变换得到（target 相对 link）
R_target = R_target_in_link
o_target = t

add_frame(fig, R_link, o_link, name='link', axis_len=0.10)
add_frame(fig, R_target, o_target, name='target', axis_len=0.10)

# 两个原点之间连线
fig.add_trace(go.Scatter3d(
    x=[o_link[0], o_target[0]],
    y=[o_link[1], o_target[1]],
    z=[o_link[2], o_target[2]],
    mode='lines',
    line=dict(color='black', width=5, dash='dash'),
    name='link-origin -> target-origin',
    showlegend=True,
))

fig.update_layout(
    title='pose_in_link 空间关系（link 与 target 坐标系）',
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z',
        aspectmode='data',
    ),
    legend=dict(itemsizing='constant'),
    margin=dict(l=0, r=0, b=0, t=45),
)

fig.show()

## 4) 解释：这个 pose 表示的空间关系

`pose_in_link = [tx, ty, tz, qx, qy, qz, qw]` 表示 **target 坐标系相对 link 坐标系** 的位姿：

- 平移 `[tx, ty, tz]` 给出 `target` 原点在 `link` 坐标系下的位置；
- 四元数 `[qx, qy, qz, qw]` 给出 `target` 坐标轴相对 `link` 坐标轴的旋转；
- 组合得到的 `T_target_in_link` 可将 `target` 坐标中的点变换到 `link` 坐标中。

在图中：`link` 坐标系位于世界原点，`target` 坐标系位于平移向量指定位置并按四元数旋转；黑色虚线连接两个原点。